# Instalações e importações

In [1]:
# Instalações
# Remova os comentários caso seja a primeira execução

# %pip install pandas

In [2]:
from pathlib import Path
import os
import glob
import pandas as pd

# Pre processamento

In [3]:
DATA_PATH = Path().cwd() / 'data'
DATASET_PATH  = DATA_PATH / 'CSE-CIC-IDS2018' 
SAMPLE_PCT = 1                # Substitua pela porcentagem desejada (ex: 10 para 10%)
RANDOM_SEED = 1
SKIP_IF_SAMPLE_ALREADY_EXSISTS = True
SKIP_IF_PREPROCESSED_EXISTS = True

SAMPLES_PATH = DATASET_PATH / 'samples'
PROCESSEDS_PATH = DATASET_PATH / 'processed'
os.makedirs(SAMPLES_PATH, exist_ok=True)
os.makedirs(PROCESSEDS_PATH, exist_ok=True)

processed_file_path = PROCESSEDS_PATH / f'data-{SAMPLE_PCT}_pct.csv'
if processed_file_path.is_file():
    df = pd.read_csv(processed_file_path)

### Cria samples dos arquivos originais que são muito grandes

In [4]:
if (not processed_file_path.is_file() or not SKIP_IF_PREPROCESSED_EXISTS):

    csv_files_paths = glob.glob(os.path.join(DATASET_PATH, '*.csv'))

    if not csv_files_paths:
        print(f"Nenhum arquivo .csv encontrado no diretório: {DATASET_PATH}")
    else:
        print(f"Encontrados {len(csv_files_paths)} arquivos. Iniciando amostragem de {SAMPLE_PCT}%...\n")
        
        for file_path in csv_files_paths:
            filename = os.path.basename(file_path)
            filename_no_ext, ext = os.path.splitext(filename)
            
            new_filename = f"{filename_no_ext}-{SAMPLE_PCT}_pct{ext}"
            save_path = SAMPLES_PATH / new_filename
            
            if (save_path.is_file() and SKIP_IF_SAMPLE_ALREADY_EXSISTS):
                print(f"⏩ Arquivo {new_filename} já existente, processamento ignorado!")
                continue

            try:
                df = pd.read_csv(file_path)
                sample_df = df.sample(frac=SAMPLE_PCT / 100.0, random_state=RANDOM_SEED)
                sample_df.to_csv(save_path, index=False)
                
                print(f"✅ Salvo: {new_filename} | Linhas originais: {len(df)} -> Amostra: {len(sample_df)}")
                
            except Exception as e:
                print(f"❌ Erro ao processar o arquivo {filename}: {e}")

        print("\nProcessamento concluído com sucesso!")
else:
    print("⏩ Célula ignorada, dataset pre-processado encontrado")

Encontrados 10 arquivos. Iniciando amostragem de 1%...

⏩ Arquivo 02-21-2018-1_pct.csv já existente, processamento ignorado!
⏩ Arquivo 02-15-2018-1_pct.csv já existente, processamento ignorado!
⏩ Arquivo 02-14-2018-1_pct.csv já existente, processamento ignorado!
⏩ Arquivo 03-01-2018-1_pct.csv já existente, processamento ignorado!
⏩ Arquivo 03-02-2018-1_pct.csv já existente, processamento ignorado!
⏩ Arquivo 02-22-2018-1_pct.csv já existente, processamento ignorado!
⏩ Arquivo 02-16-2018-1_pct.csv já existente, processamento ignorado!
✅ Salvo: 02-20-2018-1_pct.csv | Linhas originais: 7948748 -> Amostra: 79487
✅ Salvo: 02-23-2018-1_pct.csv | Linhas originais: 1048575 -> Amostra: 10486


/tmp/ipykernel_8634/870203445.py:22: DtypeWarning: Columns (0: Dst Port, 1: Protocol, 2: Flow Duration, 3: Tot Fwd Pkts, 4: Tot Bwd Pkts, 5: TotLen Fwd Pkts, 6: TotLen Bwd Pkts, 7: Fwd Pkt Len Max, 8: Fwd Pkt Len Min, 9: Fwd Pkt Len Mean, 10: Fwd Pkt Len Std, 11: Bwd Pkt Len Max, 12: Bwd Pkt Len Min, 13: Bwd Pkt Len Mean, 14: Bwd Pkt Len Std, 15: Flow Byts/s, 16: Flow Pkts/s, 17: Flow IAT Mean, 18: Flow IAT Std, 19: Flow IAT Max, 20: Flow IAT Min, 21: Fwd IAT Tot, 22: Fwd IAT Mean, 23: Fwd IAT Std, 24: Fwd IAT Max, 25: Fwd IAT Min, 26: Bwd IAT Tot, 27: Bwd IAT Mean, 28: Bwd IAT Std, 29: Bwd IAT Max, 30: Bwd IAT Min, 31: Fwd PSH Flags, 32: Bwd PSH Flags, 33: Fwd URG Flags, 34: Bwd URG Flags, 35: Fwd Header Len, 36: Bwd Header Len, 37: Fwd Pkts/s, 38: Bwd Pkts/s, 39: Pkt Len Min, 40: Pkt Len Max, 41: Pkt Len Mean, 42: Pkt Len Std, 43: Pkt Len Var, 44: FIN Flag Cnt, 45: SYN Flag Cnt, 46: RST Flag Cnt, 47: PSH Flag Cnt, 48: ACK Flag Cnt, 49: URG Flag Cnt, 50: CWE Flag Count, 51: ECE Flag C

✅ Salvo: 02-28-2018-1_pct.csv | Linhas originais: 613104 -> Amostra: 6131

Processamento concluído com sucesso!


### Unifica todas as amostras em um único dataframe

In [ ]:
if (not processed_file_path.is_file() or not SKIP_IF_PREPROCESSED_EXISTS):
    sampled_csv_files = glob.glob(os.path.join(SAMPLES_PATH, '*.csv'))

    # Identificando as colunas padrões de todos os arquivos
    base_columns = None
    for file in sampled_csv_files:
        # Lê apenas a primeira linha para ser ultra rápido e checar o número de colunas
        df_temp = pd.read_csv(file, nrows=0)
        if len(df_temp.columns) == 80:
            base_columns = list(df_temp.columns)
            print(f"📌 Padrão de 80 colunas encontrado e definido a partir de: {os.path.basename(file)}")
            break

    # Validação caso nenhum arquivo tenha 80 colunas
    if base_columns is None:
        raise ValueError("❌ Nenhum arquivo com exatamente 80 colunas foi encontrado para servir de base.")

    # Lê, filtra colunas e unifica os datasets
    df_list: list[pd.DataFrame] = []

    print("\nIniciando a leitura e unificação dos arquivos...")
    for file in sampled_csv_files:
        filename = os.path.basename(file)
        try:
            # Lê o arquivo completo
            df = pd.read_csv(file)
            
            # Se o arquivo tiver colunas a mais (ex: 84 colunas), filtramos apenas as 80 da base
            # Se tiver exatamente 80, ele mantém igual.
            filtered_df: pd.DataFrame = df[base_columns]
            df_list.append(filtered_df)
            print(f"   ✅ {filename} processado ({len(df.columns)} colunas -> reduzido para 80)")
            
        except KeyError:
            print(f"   ⚠️ Avisando: O arquivo {filename} não possui todas as colunas necessárias da base e foi ignorado.")
        except Exception as e:
            print(f"   ❌ Erro ao ler {filename}: {e}")

    # Consolida tudo em um único DataFrame
    df = pd.concat(df_list, ignore_index=True)

    print("\nConvertendo a coluna 'Timestamp' para formato Epoch...")

    # Remove linhas onde o valor é literalmente a palavra 'Timestamp' (cabeçalhos no meio do dado)
    df = df[df['Timestamp'] != 'Timestamp'].copy()

    # 1. Transforma o texto em um objeto de data/hora do pandas (datetime)
    df['Timestamp_dt'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S')

    # 2. Converte o datetime para nanosegundos desde a epoch, e divide por 10**9 para virar segundos (Timestamp Unix)
    # Convertemos para 'int64' para remover as casas decimais
    df['Timestamp'] = (df['Timestamp_dt'].astype('int64') // 10**9)

    # 3. Remove a coluna temporária de datetime que criamos para o cálculo
    df = df.drop(columns=['Timestamp_dt'])

    # 4. Ordena o dataframe de forma cronológica (do mais antigo para o mais recente)
    df = df.sort_values(by='Timestamp').reset_index(drop=True)

    df.to_csv(processed_file_path, index=False)
    print(f"✨ Concluído! Dataset final gerado com {df.shape[0]} linhas e {df.shape[1]} colunas.")
else: 
    print("⏩ Célula ignorada, dataset pre-processado encontrado")
df.head(5)

📌 Padrão de 80 colunas encontrado e definido a partir de: 02-15-2018-1_pct.csv

Iniciando a leitura e unificação dos arquivos...
   ✅ 02-15-2018-1_pct.csv processado (80 colunas -> reduzido para 80)
   ✅ 02-14-2018-1_pct.csv processado (80 colunas -> reduzido para 80)
   ✅ 02-28-2018-1_pct.csv processado (80 colunas -> reduzido para 80)
   ✅ 03-02-2018-1_pct.csv processado (80 colunas -> reduzido para 80)
   ✅ 02-23-2018-1_pct.csv processado (80 colunas -> reduzido para 80)
   ✅ 02-22-2018-1_pct.csv processado (80 colunas -> reduzido para 80)
   ✅ 02-20-2018-1_pct.csv processado (84 colunas -> reduzido para 80)
   ✅ 03-01-2018-1_pct.csv processado (80 colunas -> reduzido para 80)
   ✅ 02-21-2018-1_pct.csv processado (80 colunas -> reduzido para 80)
   ✅ 02-16-2018-1_pct.csv processado (80 colunas -> reduzido para 80)

Convertendo a coluna 'Timestamp' para formato Epoch...
✨ Concluído! Dataset final gerado com 162331 linhas e 80 colunas.


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,53,17,1518570,337,1,1,38.0,54.0,38.0,38.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,51289,6,1518570,45,1,1,0.0,0.0,0.0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,53,17,1518570,939,1,1,35.0,51.0,35.0,35.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,53,17,1518570,19292,1,1,36.0,107.0,36.0,36.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,53,17,1518570,156411,2,2,84.0,194.0,42.0,42.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


#### Limpeza dos dados

In [6]:
before_lines = df.shape[0]
df = df.dropna()
after_lines = df.shape[0]
print(f"🗑️ Linhas com valores nulos removidas: {before_lines - after_lines}")

before_cols = df.shape[1]
df = df.loc[:, df.nunique() > 1]
after_cols = df.shape[1]

print(f"🗑️ Colunas constantes removidas: {before_cols - after_cols}")

print(f"\n✨ Limpeza concluída! Dataset atualizado com {df.shape[0]} linhas e {df.shape[1]} colunas.")

df.head(5)

🗑️ Linhas com valores nulos removidas: 602
🗑️ Colunas constantes removidas: 8

✨ Limpeza concluída! Dataset atualizado com 161729 linhas e 72 colunas.


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,53,17,1518570,337,1,1,38.0,54.0,38.0,38.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,51289,6,1518570,45,1,1,0.0,0.0,0.0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,53,17,1518570,939,1,1,35.0,51.0,35.0,35.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,53,17,1518570,19292,1,1,36.0,107.0,36.0,36.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,53,17,1518570,156411,2,2,84.0,194.0,42.0,42.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [36]:
numerical_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(exclude='number').columns.tolist()

print(f"📊 COLUNAS NUMÉRICAS ({len(numerical_cols)} no total):")
if numerical_cols:
    print(', '.join(numerical_cols))
else:
    print("Nenhuma coluna numérica encontrada.")

print("\n" + "-"*80 + "\n")

print(f"🔤 COLUNAS CATEGÓRICAS / TEXTUAIS ({len(categorical_cols)} no total):")
if categorical_cols:
    print(', '.join(categorical_cols))
else:
    print("Nenhuma coluna categórica ou textual encontrada.")

📊 COLUNAS NUMÉRICAS (71 no total):
Dst Port, Protocol, Timestamp, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Fwd Pkt Len Max, Fwd Pkt Len Min, Fwd Pkt Len Mean, Fwd Pkt Len Std, Bwd Pkt Len Max, Bwd Pkt Len Min, Bwd Pkt Len Mean, Bwd Pkt Len Std, Flow Byts/s, Flow Pkts/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Tot, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Bwd IAT Tot, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Fwd URG Flags, Fwd Header Len, Bwd Header Len, Fwd Pkts/s, Bwd Pkts/s, Pkt Len Min, Pkt Len Max, Pkt Len Mean, Pkt Len Std, Pkt Len Var, FIN Flag Cnt, SYN Flag Cnt, RST Flag Cnt, PSH Flag Cnt, ACK Flag Cnt, URG Flag Cnt, CWE Flag Count, ECE Flag Cnt, Down/Up Ratio, Pkt Size Avg, Fwd Seg Size Avg, Bwd Seg Size Avg, Subflow Fwd Pkts, Subflow Fwd Byts, Subflow Bwd Pkts, Subflow Bwd Byts, Init Fwd Win Byts, Init Bwd Win Byts, Fwd Act Data Pkts, Fwd Seg Size Min, Active Mean, Active Std, A

### Correção de Tipos: Convertendo Falsas Numéricas em Categóricas

In [ ]:
print("Corrigindo os tipos de dados das colunas...\n")

# Lista das colunas que o Pandas acha que são números, mas na verdade são categorias nominais
false_numerical_cols = ['Dst Port', 'Protocol']

# Filtramos apenas as que realmente estão no dataframe (caso alguma tenha sido removida nas etapas de limpeza)
cols_to_convert = [col for col in false_numerical_cols if col in df.columns]

if cols_to_convert:
    for col in cols_to_convert:
        df[col] = df[col].astype('category')
        print(f"🔄 Coluna '{col}' convertida com sucesso para o tipo 'category'.")
else:
    print("Nenhuma das colunas falsas numéricas foi encontrada no DataFrame atual.")

# Exibe o tipo de dado atualizado apenas das colunas que modificamos, além do nosso Timestamp
check_cols = cols_to_convert + (['Timestamp'] if 'Timestamp' in df.columns else [])
print("\nTipos de dados atualizados para conferência:")
print(df[check_cols].dtypes)

Corrigindo os tipos de dados das colunas...

🔄 Coluna 'Dst Port' convertida com sucesso para o tipo 'category'.
🔄 Coluna 'Protocol' convertida com sucesso para o tipo 'category'.

Tipos de dados atualizados para conferência:
Dst Port     category
Protocol     category
Timestamp       int64
dtype: object


# Algoritmo Nayve